In [46]:
from langchain.tools import tool
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.messages.tool import ToolMessage
from typing import TypedDict, List, Any, Optional
from langgraph.graph import StateGraph, START, END
import getpass

OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

# -- 1) Define our single business tool
@tool
def cancel_order(order_id: str) -> str:
    """Cancel an order that hasn't shipped."""
    # (Here you'd call your real backend API)
    return f"Order {order_id} has been cancelled."

# -- 2) The agent "brain": invoke LLM, run tool, then invoke LLM again
def call_model(state):
    msgs = state["messages"]
    order = state.get("order", {"order_id": "UNKNOWN"})
    # System prompt tells the model exactly what to do
    prompt = (
        f'''You are an ecommerce support agent.
        ORDER ID: {order['order_id']}
        If the customer asks to cancel, call cancel_order(order_id)
        and then send a simple confirmation.
        Otherwise, just respond normally.'''
    )
    full = [SystemMessage(prompt)] + msgs
    
    # 1st LLM pass: decides whether to call our tool
    AIMessage = ChatOpenAI(
        openai_api_key=OPENAI_API_KEY,
        model="gpt-5-nano", temperature=0)
    resp = AIMessage.invoke(full)
    first = resp.content
    out = [first]
    
    if getattr(first, "tool_calls", None):
        # run the cancel_order tool
        tc = first.tool_calls[0]
        result = cancel_order(**tc["args"])
        out.append(ToolMessage(content=result, tool_call_id=tc["id"]))
        
        # 2nd LLM pass: generate the final confirmation text
        AIMessage = ChatOpenAI(
            openai_api_key=OPENAI_API_KEY,
            model="gpt-5-nano", temperature=0)(full + out)
        resp = AIMessage.invoke(full + out)
        second = resp.content
        out.append(second)
    return {"messages": out}

def ExtractResult(result):
    msgs = []
    for msg in result["messages"]:
        if isinstance(msg, BaseMessage):
            msgs.append(msg.content)
        else:
        # กรณีที่หลุดมาเป็น string ธรรมดา
            msgs.append(msg)
    return ".\n".join(msgs)

# -- 3) Wire it all up in a StateGraph
class State(TypedDict):
    order: Optional[Any]
    messages: list
    
def construct_graph():
    # 2. ส่งคลาส State เข้าไปแทนที่ dict ปกติ
    g = StateGraph(State) 
    g.add_node("assistant", call_model)
    g.add_edge(START, "assistant") # หรือใช้ g.set_entry_point("assistant") ตามโค้ดเดิมของคุณ
    return g.compile()

Enter your OPENAI_API_KEY ········


In [47]:
graph = construct_graph()

example_order = {"order_id": "A12345"}
convo = [HumanMessage(content="Please cancel my order A12345.")]
result = graph.invoke({"order": example_order, "messages": convo})
msg = ExtractResult(result)

print(f"assistant (string): {msg}")

assistant (string): Cancellation completed. Order A12345 has been canceled. A confirmation has been sent to your email.


In [51]:
# Minimal evaluation check
example_order = {"order_id": "B73973"}
convo = [HumanMessage(content="Please cancel order #B73973. I found a cheaper option elsewhere.")]

result = graph.invoke({"order": example_order, "messages": convo})
msg = extractResult(result)
print(f"assistant (string): {msg}")

assert any("cancel_order" in m for m in msg, "Cancel order tool not called")
assert any("cancelled" in m.lower() for m in msg, "Confirmation message missing")

print("✅ Agent passed minimal evaluation.")

assistant (string): I've canceled order B73973 as requested. If you need anything else, just let me know.
✅ Agent passed minimal evaluation.
